# EDA y Visualización: interrogar los datos antes de modelarlos

**Ciencia de Datos, Sección A** · Sesión 9 · 20 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

## 1. El checklist del EDA

1. **Forma**: ¿cuántas filas y columnas? ¿una fila es qué cosa?
2. **Tipos**: ¿numérica, categórica, fecha, texto?
3. **Faltantes**: ¿cuántos, dónde y por qué?
4. **Distribuciones**: ¿centro, dispersión, sesgo, extremos?
5. **Relaciones**: ¿qué se mueve junto con qué?

Ninguna de las cinco es opcional.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("penguins")
print(df.shape)
df.head()

In [ ]:
df.info()          # tipos y no-nulos por columna

In [ ]:
print(df.describe())              # count, mean, std, min, cuartiles
print(df["species"].value_counts())
print(df.isna().sum())            # faltantes por columna

## 2. Datos faltantes: el mecanismo importa

- **MCAR** (completamente al azar): la falta no depende de nada.
- **MAR** (al azar dado lo observado): la falta depende de *otras* columnas que sí tenemos.
- **MNAR** (no al azar): la falta depende del **valor que falta**.

Ejemplo de la encuesta de ingresos: si los que ganan mucho no contestan *porque* ganan mucho (MNAR), imputar con la media subestima el ingreso promedio y lo reportaremos con total confianza.

El mecanismo no se lee en los datos: se razona sobre cómo se recolectaron.

In [ ]:
print(df.isna().mean().round(3))   # fracción faltante por columna

# Marcar ANTES de tocar nada: la ausencia a veces es señal
df["masa_faltaba"] = df["body_mass_g"].isna()

# Eliminar: solo si son pocos y sospechamos MCAR
print(df.dropna(subset=["sex"]).shape)

# Imputar: la mediana aguanta mejor los extremos
m = df["body_mass_g"].median()
df["body_mass_g"] = df["body_mass_g"].fillna(m)
print(df.isna().sum().sum())

**Spoiler de la próxima clase:** esa mediana debe calcularse **solo con los datos de entrenamiento**. Calcularla sobre todo el dataset es *leakage*, y el martes veremos por qué eso infla los resultados.

## 3. Outliers: detectarlos y decidir

El IQR usa cuartiles, así que los extremos no mueven el criterio. El z-score usa media y desviación, que sí los mueven: un outlier gigante infla `std` y termina escondiéndose a sí mismo.

In [ ]:
x = df["body_mass_g"].dropna()

q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
bajo, alto = q1 - 1.5*iqr, q3 + 1.5*iqr
print(bajo, alto)
print(x[(x < bajo) | (x > alto)])   # regla del boxplot

z = (x - x.mean()) / x.std()
print(x[z.abs() > 3])              # regla z-score

# Ojo: penguins es limpio y ninguna de las dos reglas marca nada.
# Que NO salgan outliers también es un hallazgo del EDA.

In [ ]:
# El outlier que se esconde a sí mismo inflando la std
y = np.array([10, 11, 12, 10, 11, 5000.0])
zz = (y - y.mean()) / y.std()
print(zz.round(2))   # el 5000 no llega a |z| > 3

**¿Error o señal?** Solo hay dos respuestas: o el dato es imposible (edad 400, precio negativo) y se corrige, o es real y raro, en cuyo caso probablemente sea lo más interesante del dataset. Fraude, fallas y fugas viven en la cola de la distribución.

## 4. Distribuciones, colas largas y correlación

In [ ]:
precios = np.array([100, 120, 150, 90, 8000.0])
print(precios.mean())      # 1692: lo arrastró el extremo
print(np.median(precios))  # 120: resiste
print(np.log1p(precios).round(2))   # comprime la cola derecha

In [ ]:
# Anscombe: mismos estadísticos, cuatro formas distintas
ans = sns.load_dataset("anscombe")
print(ans.groupby("dataset").agg(
    media_x=("x", "mean"),
    media_y=("y", "mean"),
    std_y=("y", "std"),
).round(2))

for nombre, g in ans.groupby("dataset"):
    print(nombre, round(g["x"].corr(g["y"]), 3))   # todos ~0.816

In [ ]:
sns.lmplot(ans, x="x", y="y", col="dataset", col_wrap=2, height=2.5)
plt.show()   # cuatro historias completamente distintas

## 5. Matplotlib para pulir, Seaborn para explorar

Eviten el estilo `plt.plot()` suelto: funciona en el notebook y se rompe apenas necesiten dos paneles. Con `fig, ax` tienen objetos explícitos.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["bill_length_mm"], df["body_mass_g"])
ax.set_xlabel("Largo del pico (mm)")
ax.set_ylabel("Masa (g)")
ax.set_title("Picos largos, cuerpos pesados")
fig.tight_layout()
plt.show()

In [ ]:
sns.histplot(df, x="body_mass_g", hue="species")
plt.show()

sns.boxplot(df, x="species", y="body_mass_g")
plt.show()

sns.scatterplot(df, x="bill_length_mm", y="body_mass_g", hue="species")
plt.show()

In [ ]:
num = df.select_dtypes("number")
sns.heatmap(num.corr(), annot=True, cmap="vlag")
plt.show()

### ¿Qué gráfica uso?

| Pregunta | Gráfica | Evitar |
|---|---|---|
| ¿Cuál es mayor? | barras | pie con 8 rebanadas |
| ¿Cómo se distribuye? | histograma, box | barra del promedio |
| ¿Se mueven juntas? | scatter | barras apiladas |
| ¿Cómo evoluciona? | línea | barras por año |
| ¿Qué parte del todo? | barra apilada, treemap | pie 3D |

Regla práctica: si tienen que explicar cómo leer la gráfica antes de explicar el hallazgo, elijan otra gráfica.

## 6. Storytelling y ética

- **Título que afirma**, no que describe: "Occidente cayó 22% mientras el resto creció", no "Ventas por región".
- **Anoten el punto clave** en la gráfica misma.
- **Quiten tinta inútil**: rejillas pesadas, bordes, fondos, decimales que nadie usa.
- **Una gráfica, una idea.**

Formas técnicamente correctas de mentir: eje truncado, ventana temporal escogida, doble eje ajustado hasta que las curvas coincidan, áreas mal escaladas, eje invertido.

## 7. Ejercicios

Completen donde dice `# ¿Qué va aquí?`.

### Ejercicio 1: EDA guiado

Corran el checklist completo sobre `penguins` y escriban **tres hallazgos** en texto, no en gráficas.

In [ ]:
df = sns.load_dataset("penguins")   # versión original, sin tocar

# a) forma, tipos y faltantes
# ¿Qué va aquí?

# b) distribución de body_mass_g por species

# c) la relación más fuerte entre dos numéricas

# d) escriban 3 hallazgos en la celda markdown de abajo

**Mis tres hallazgos:**

1. 
2. 
3. 

### Ejercicio 2: clasificar los faltantes

No hay respuesta única. Lo que se evalúa es el argumento.

In [ ]:
print(df.isna().sum())

# a) ¿qué columnas tienen faltantes y cuántos?

# b) ¿las filas con sex faltante son distintas del resto
#    en masa o en especie? (comparen medias y proporciones)
sin_sexo = ...
con_sexo = ...

# c) argumenten: ¿MCAR, MAR o MNAR? Justifiquen abajo

# d) propongan una estrategia y digan qué sesgo introduce

### Ejercicio 3: la gráfica mentirosa

Construyan la versión tramposa y la honesta del mismo dato.

In [ ]:
ventas = [100.0, 101.2, 100.8, 102.0, 101.5]
anios = [2021, 2022, 2023, 2024, 2025]

# a) versión mentirosa: eje y de 100 a 102
# ¿Qué va aquí?  Pista: ax.set_ylim(100, 102)

# b) versión honesta: eje y desde 0

# c) ¿qué título pondría cada uno?
#    Mentiroso: "..."
#    Honesto:   "..."

## Lo esencial de hoy

- EDA es un **checklist**: forma, tipos, faltantes, distribuciones, relaciones
- El **mecanismo** del faltante (MCAR, MAR, MNAR) decide qué se puede hacer con él
- Un outlier es **error o señal**: hay que decidir cuál, no borrarlo por reflejo
- Colas largas: mediana e IQR, y el `log` como aliado
- **Nunca un $r$ sin su scatter** (Anscombe, Datasaurus)
- Seaborn para explorar, Matplotlib para pulir, título que afirma

**Próxima clase (martes 25): Feature Engineering.** Recuerden: la HDT 3 se entrega ese mismo martes.